## Embedding and Chat with gemini

In [1]:
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [6]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [7]:
embeddings.embed_query("Hello world")

[-0.02342152,
 0.01676572,
 0.009261323,
 -0.06383,
 -0.0026262768,
 0.0010187156,
 -0.01125684,
 0.0130036585,
 0.008751671,
 0.0012745215,
 -0.0007880878,
 -0.019086141,
 0.030971918,
 0.044264916,
 0.11137619,
 0.018025596,
 -0.0031383373,
 -0.0130702155,
 0.0054121013,
 -0.009036983,
 0.009904047,
 -0.010718052,
 0.012551899,
 0.011420718,
 -0.03375867,
 0.008877066,
 0.027795823,
 0.0037452083,
 0.029841818,
 0.019861836,
 0.018075233,
 -0.007178086,
 -0.02497957,
 0.011720823,
 -0.0073324037,
 0.0068515264,
 0.011618152,
 0.004278904,
 -0.0099426275,
 -0.009696086,
 -0.0037028084,
 0.006349137,
 -0.012098703,
 -0.015629271,
 -0.022799378,
 -0.012305125,
 -0.01497548,
 -0.006503783,
 -0.005624279,
 0.020265369,
 -0.029982245,
 -0.008486281,
 -0.0050998786,
 -0.14985937,
 -0.017222198,
 0.011674307,
 -0.014542328,
 0.02246327,
 -0.009126675,
 -0.018621653,
 -0.004780002,
 0.0020153068,
 -0.0133157885,
 -0.006439805,
 0.0008783784,
 -0.023549458,
 -0.008333907,
 0.01863065,
 -0.0190

In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

llm.invoke("hi how are you?").content[0]["text"]

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


"I'm doing great, thank you for asking! How are you doing today? Is there anything I can help you with?"

## It is simple AI Assistant without RAG Pipelie

In [13]:
while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        print("Exiting the chat. Goodbye!")
        break
    response = llm.invoke(user_input).content[0]["text"]
    print(f"User: {user_input.lower()}")
    print(f"AI: {response}")

User: cam you tell me in which year this deepseek-v4 model..?
AI: As of right now, **there is no "DeepSeek-V4" model.**

The current state-of-the-art models from DeepSeek are part of the **DeepSeek-V3** series, which was released in **December 2024**.

If you have seen references to a "V4" model, it is likely a misunderstanding or speculation, as DeepSeek has not announced or released a version with that designation yet.
Exiting the chat. Goodbye!


Note: It saying wrong like 2024 but actually in reasearch paper it is 2026. In this senarios we required RAG system to stop LLM hallisunation

## Rag pipeline implementation

In [23]:
##---------------------##
## Data loader 
##---------------------##

from langchain_community.document_loaders import PyPDFLoader
pdf_loader = PyPDFLoader("deepseek-v4-2026.pdf")
pdf_documents = pdf_loader.load()

In [24]:
len(pdf_documents)

58

In [25]:
print(pdf_documents[0].page_content[:500])   # Display the first 500 characters of the first page's content

DeepSeek-V4:
Towards Highly Efficient Million-Token Context Intelligence
DeepSeek-AI
research@deepseek.com
Abstract
We present a preview version of DeepSeek-V4 series, including two strong Mixture-of-
Experts (MoE) language models — DeepSeek-V4-Pro with 1.6T parameters (49B activated) and
DeepSeek-V4-Flash with 284B parameters (13B activated) — both supporting a context length of
one million tokens. DeepSeek-V4 series incorporate several key upgrades in architecture and op-
timization: (1) a hyb


In [26]:
##---------------------##
## chuncking the document into smaller pieces
##---------------------##
from langchain.text_splitter import RecursiveCharacterTextSplitter
chunker =RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=400)
chunked_data = chunker.split_documents(pdf_documents)

In [27]:
len(chunked_data)

283

In [28]:
chunked_data[0].page_content 

'DeepSeek-V4:\nTowards Highly Efficient Million-Token Context Intelligence\nDeepSeek-AI\nresearch@deepseek.com\nAbstract\nWe present a preview version of DeepSeek-V4 series, including two strong Mixture-of-\nExperts (MoE) language models — DeepSeek-V4-Pro with 1.6T parameters (49B activated) and\nDeepSeek-V4-Flash with 284B parameters (13B activated) — both supporting a context length of\none million tokens. DeepSeek-V4 series incorporate several key upgrades in architecture and op-\ntimization: (1) a hybrid attention architecture that combines Compressed Sparse Attention (CSA)\nand Heavily Compressed Attention (HCA) to improve long-context efficiency; (2) Manifold-\nConstrained Hyper-Connections (mHC) that enhance conventional residual connections; (3)\nand the Muon optimizer for faster convergence and greater training stability. We pre-train\nboth models on more than 32T diverse and high-quality tokens, followed by a comprehensive'

In [29]:
chunked_data[0].metadata

{'producer': 'pikepdf 8.15.1',
 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)',
 'creationdate': '',
 'author': 'DeepSeek-AI; Anyi Xu; Bangcai Lin; Bing Xue; Bingxuan Wang; Bingzheng Xu; Bochao Wu; Bowei Zhang; Chaofan Lin; Chen Dong; Chenchen Ling; Chengda Lu; Chenggang Zhao; Chengqi Deng; Chengyu Hou; Chenhao Xu; Chenze Shao; Chong Ruan; Conner Sun; Damai Dai; Daya Guo; Dejian Yang; Deli Chen; Donghao Li; Dongjie Ji; Erhang Li; Fang Wei; Fangyun Lin; Fangzhou Yuan; Feiyu Xia; Fucong Dai; Guangbo Hao; Guanting Chen; Guoai Cao; Guolai Meng; Guowei Li; Han Yu; Han Zhang; Hanwei Xu; Hao Li; Haofen Liang; Haoling Zhang; Haoming Luo; Haoran Wei; Haotian Yuan; Haowei Zhang; Haowen Luo; Haoyu Chen; Haozhe Ji; Hengqing Zhang; Honghui Ding; Hongxuan Tang; Huanqi Cao; Huazuo Gao; Hui Qu; Hui Zeng; J Yang; JQ Zhu; Jia Luo; Jia Song; Jia Yu; Jialiang Huang; Jialu Cai; Jian Liang; Jiangting Zhou; Jiasheng Ye; Jiashi Li; Jiaxin Xu; Jiewen Hu; Jieyu Yang; Jin Chen; Jin Yan; Jingchang Chen; Jingli Zhou;

In [9]:
##---------------------##
## Vector Store Creation
##---------------------##
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

index = faiss.IndexFlatL2(384)

from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)


In [10]:
ids = [str(i) for i in range(len(chunked_data))]
vector_store.add_documents(chunked_data, ids=ids)

GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-1.0\nPlease retry in 9.45137351s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier', 'q

In [30]:
import numpy as np

emb = embeddings.embed_query("test")
print("embedding dim:", np.array(emb).shape)   # expected (D,)
print("index.d:", getattr(index, "d", None))   # IndexFlatL2 has .d

embedding dim: (384,)
index.d: 384


In [31]:
##---------------------##
## Vector Store Creation
##---------------------##
# This is an alternative vector store creation using a different embedding model from HuggingFace.
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

index = faiss.IndexFlatL2(384)  # Adjust the dimension based on your embedding model

from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

In [32]:
ids = [str(i) for i in range(len(chunked_data))]
vector_store.add_documents(chunked_data, ids=ids) # As of now data store in ram

['0',
 '1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '9',
 '10',
 '11',
 '12',
 '13',
 '14',
 '15',
 '16',
 '17',
 '18',
 '19',
 '20',
 '21',
 '22',
 '23',
 '24',
 '25',
 '26',
 '27',
 '28',
 '29',
 '30',
 '31',
 '32',
 '33',
 '34',
 '35',
 '36',
 '37',
 '38',
 '39',
 '40',
 '41',
 '42',
 '43',
 '44',
 '45',
 '46',
 '47',
 '48',
 '49',
 '50',
 '51',
 '52',
 '53',
 '54',
 '55',
 '56',
 '57',
 '58',
 '59',
 '60',
 '61',
 '62',
 '63',
 '64',
 '65',
 '66',
 '67',
 '68',
 '69',
 '70',
 '71',
 '72',
 '73',
 '74',
 '75',
 '76',
 '77',
 '78',
 '79',
 '80',
 '81',
 '82',
 '83',
 '84',
 '85',
 '86',
 '87',
 '88',
 '89',
 '90',
 '91',
 '92',
 '93',
 '94',
 '95',
 '96',
 '97',
 '98',
 '99',
 '100',
 '101',
 '102',
 '103',
 '104',
 '105',
 '106',
 '107',
 '108',
 '109',
 '110',
 '111',
 '112',
 '113',
 '114',
 '115',
 '116',
 '117',
 '118',
 '119',
 '120',
 '121',
 '122',
 '123',
 '124',
 '125',
 '126',
 '127',
 '128',
 '129',
 '130',
 '131',
 '132',
 '133',
 '134',
 '135',
 '136',
 '137',
 '138'

In [1]:
##---------------------##
## Retriver
##---------------------##
retriever = vector_store.as_retriever(search_kwargs={"k": 10})

NameError: name 'vector_store' is not defined

In [34]:
retriever.invoke("what is this deepseek-v4-2026 model about?")

[Document(id='139', metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '', 'author': 'DeepSeek-AI; Anyi Xu; Bangcai Lin; Bing Xue; Bingxuan Wang; Bingzheng Xu; Bochao Wu; Bowei Zhang; Chaofan Lin; Chen Dong; Chenchen Ling; Chengda Lu; Chenggang Zhao; Chengqi Deng; Chengyu Hou; Chenhao Xu; Chenze Shao; Chong Ruan; Conner Sun; Damai Dai; Daya Guo; Dejian Yang; Deli Chen; Donghao Li; Dongjie Ji; Erhang Li; Fang Wei; Fangyun Lin; Fangzhou Yuan; Feiyu Xia; Fucong Dai; Guangbo Hao; Guanting Chen; Guoai Cao; Guolai Meng; Guowei Li; Han Yu; Han Zhang; Hanwei Xu; Hao Li; Haofen Liang; Haoling Zhang; Haoming Luo; Haoran Wei; Haotian Yuan; Haowei Zhang; Haowen Luo; Haoyu Chen; Haozhe Ji; Hengqing Zhang; Honghui Ding; Hongxuan Tang; Huanqi Cao; Huazuo Gao; Hui Qu; Hui Zeng; J Yang; JQ Zhu; Jia Luo; Jia Song; Jia Yu; Jialiang Huang; Jialu Cai; Jian Liang; Jiangting Zhou; Jiasheng Ye; Jiashi Li; Jiaxin Xu; Jiewen Hu; Jieyu Yang; Jin Chen; Jin Yan; Ji

In [35]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [37]:
retriever.invoke("what is a release date of deepseek v4 research paper?")[0].page_content

'Wang, Peng Zhang, Ruyi Chen, Shangmian Sun, Shaoqing Wu, W.L. Xiao, Wei An, Wenqing\nHou, Xianzu Wang, Xiaowen Sun, Xiaoxiang Wang, Xinyu Zhang, Xueyin Chen, Yao Xu, Yi\nShao, Yiling Ma, Ying Tang, Yuehan Yang, Yuer Xu, Yukun Zha, Yuping Lin, Yuting Yan, Zekai\nZhang, Zhe Ju, Zheren Gao, Zhongyu Wu, Zihua Qu, Ziyi Wan.\nA.2. Acknowledgment\nWe would like to thank Dolly Deng and other testers for their valuable suggestions and feedback\nregarding the capabilities of DeepSeek-V4 series models.\nB. Evaluation Details\nTable 9|Agentic Search vs. Retrieval Augmented Search for DeepSeek-V4-Pro.\nDifficulty Category # Agent Win RAG Win Tie Agent% RAG% Tie%\nEasy Objective Q&A (客观问答) 196 110 43 43 56.1 21.9 21.9\nSubjective Q&A (主观问答) 321 198 56 67 61.7 17.4 20.9\nHard Objective Q&A (客观问答) 168 102 33 33 60.7 19.6 19.6\nSubjective Q&A (主观问答) 184 126 27 31 68.5 14.7 16.8\nTotal (总计) 869 536 159 174 61.7 18.3 20.0\nTable 10 | Cost Comparison:Agentic Search vs. Retrieval Augmented Search (Mean) f

In [38]:
retriever.invoke("what is a release date of deepseek v4 research paper?")[1].page_content

'HLE w/ tools(Pass@1) 53.1 52.0 51.6 54.050.4 48.2\nGDPval-AA(Elo) 1619 16741314 1482 1535 1554\nMCPAtlas Public(Pass@1) 73.867.2 69.2 66.6 71.8 73.6\nToolathlon(Pass@1) 47.254.648.8 50.0 40.7 51.8\nKnowledge.In the evaluation of general world knowledge, DeepSeek-V4-Pro-Max, the max-\nimum reasoning effort mode of DeepSeek-V4-Pro, establishes a new state-of-the-art among\nopen-source large language models. As demonstrated by the SimpleQA-Verified, DeepSeek-V4-\nPro-Max significantly outperforms all existing open-source baselines by a margin of 20 absolute\npercentage points. Despite these advances, it currently trails the leading proprietary model,\nGemini-3.1-Pro. In the domain of educational knowledge and reasoning, DeepSeek-V4-Pro-Max\nmarginally outperforms Kimi and GLM across the MMLU-Pro, GPQA, and HLE benchmarks,\nalthough it lags behind leading proprietary models. Broadly, DeepSeek-V4-Pro-Max marks a'

In [39]:
##---------------------##
## Prompt Template
##---------------------##

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
from langchain_core.prompts import PromptTemplate
prompt=PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n{context}\n\nQuestion: {question}\n')

In [40]:
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])


In [42]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

In [43]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
rag_chain = {"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()

In [44]:
rag_chain.invoke("what is a release date of DeepSeek-V4 research paper?")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


'Based on the provided text, there is no information regarding the release date of the DeepSeek-V4 research paper.'

In [45]:
rag_chain.invoke("how many parameters are there in the DeepSeek-V4-Pro model?")

'Based on the provided context, there is no mention of the total parameter count for the DeepSeek-V4-Pro model. The text only specifies that DeepSeek-V4-Flash comprises 284B total parameters.'

In [46]:
rag_chain.invoke("how many parameters are there in the DeepSeek-V4-Flash model?")

'DeepSeek-V4-Flash comprises 284B total parameters.'

In [47]:
rag_chain.invoke("which Reinforcement Learning is used in the DeepSeek-V4 model?")

'To determine which reinforcement learning method is used in the DeepSeek-V4 model based on the provided text, I have followed this deliberation process:\n\n1.  **Analyze the Request:** The user is asking for the specific reinforcement learning method used in the DeepSeek-V4 model, restricted strictly to the provided context.\n2.  **Scan the Context for "DeepSeek-V4":** I searched the provided text for mentions of "DeepSeek-V4".\n    *   I found the following sentence: "In the post-training phase of DeepSeek-V4 series, entirely replaced by On-Policy Distillation (OPD; Gu et al., 2024; Lu and Lab, 2025)." (Note: The provided text contains a fragmented sentence structure regarding the replacement).\n3.  **Analyze the Context for "Reinforcement Learning" in relation to DeepSeek-V4:**\n    *   The text mentions: "In the post-training phase of DeepSeek-V4 series, entirely replaced by On-Policy Distillation (OPD; Gu et al., 2024; Lu and Lab, 2025)."\n    *   I checked if other sections (such